# Step 5 — Model B phase 2: projector + LoRA, indication + prior report; Model B vs Model A
**Settings:** GPU T4 x2, Internet on, `HF_TOKEN`. Inputs: as Step 4 + the Step 0, 1, 4 outputs.
The winner (pre-registered rule in `configs/step5.yaml` `compare`) carries Steps 6-10.

In [ ]:
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 0. Restore the outputs of the earlier steps (add each previous notebook's output as an input)

In [ ]:
PREV = []   # e.g. ["/kaggle/input/step0-kaggle/project", "/kaggle/input/step1-kaggle/project", ...]
os.makedirs("/kaggle/working/project", exist_ok=True)
for src in PREV:
    !cp -rn {src}/outputs /kaggle/working/project/ 2>/dev/null || true
    !cp -rn {src}/data /kaggle/working/project/ 2>/dev/null || true
!ls /kaggle/working/project/outputs/runs

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step5.yaml
!python scripts/step0_prepare_data.py --config configs/step5.yaml --smoke
!python scripts/step5_train_joint.py --run-name b2 --smoke
!python scripts/step5_generate.py --run-name b2 --smoke
!python scripts/step5_score.py --run-name b2 --smoke

## Full run (resumable)

In [ ]:
!torchrun --nproc_per_node 2 scripts/step5_train_joint.py --run-name b2
!python scripts/step5_generate.py --run-name b2
!python scripts/step5_score.py --run-name b2
!cat /kaggle/working/project/outputs/runs/step5/b2/winner.json